In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03 - Gold Layer: Reporting Aggregates
# MAGIC
# MAGIC **Goal:** produce small, fast, reporting-ready tables for BI tools / dashboards.
# MAGIC
# MAGIC Techniques demonstrated:
# MAGIC - Reads Silver **incrementally via Delta Change Data Feed (CDF)** rather than
# MAGIC   rescanning the whole table each run
# MAGIC - Recomputes aggregates only for the affected `(customer_id, transaction_date)` /
# MAGIC   `(transaction_month)` / `(category)` keys touched by the change, then **MERGEs**
# MAGIC   those keys into the Gold tables -> idempotent and cheap even as Silver grows
# MAGIC - Three Gold tables covering three common reporting angles: revenue by customer/day,
# MAGIC   revenue by month, and category/transaction-volume performance

# COMMAND ----------

from pyspark.sql import functions as F

CATALOG = "lakehouse_demo"
SCHEMA = "transactions"

SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_transactions"
CHECKPOINT_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

GOLD_CUSTOMER_DAILY = f"{CATALOG}.{SCHEMA}.gold_revenue_by_customer_daily"
GOLD_MONTHLY = f"{CATALOG}.{SCHEMA}.gold_revenue_by_month"
GOLD_CATEGORY = f"{CATALOG}.{SCHEMA}.gold_category_performance"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# COMMAND ----------

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_CUSTOMER_DAILY} (
    customer_id       STRING,
    transaction_date  DATE,
    gross_revenue     DOUBLE,
    net_revenue       DOUBLE,
    txn_count         BIGINT,
    refund_count      BIGINT,
    last_updated_ts   TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_MONTHLY} (
    transaction_month STRING,
    gross_revenue      DOUBLE,
    net_revenue         DOUBLE,
    txn_count            BIGINT,
    distinct_customers    BIGINT,
    refund_count           BIGINT,
    last_updated_ts        TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_CATEGORY} (
    category           STRING,
    transaction_month  STRING,
    net_revenue         DOUBLE,
    txn_count             BIGINT,
    avg_txn_value           DOUBLE,
    last_updated_ts          TIMESTAMP
) USING DELTA
""")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Control table for incremental tracking
# MAGIC
# MAGIC NOTE: `writeStream.foreachBatch(...)` is not usable here on Databricks Free Edition.
# MAGIC Databricks automatically wraps every `foreachBatch` call in an internal caching
# MAGIC optimisation (`addBatchOptimized`) that calls `.rdd` under the hood, and `.rdd` is not
# MAGIC implemented under Spark Connect (the engine Free Edition / serverless notebooks run
# MAGIC on) - this fails with `PySparkNotImplementedError: [NOT_IMPLEMENTED] rdd is not
# MAGIC implemented`, regardless of what the foreachBatch function itself does, and there is no
# MAGIC config available on Free Edition to disable it.
# MAGIC
# MAGIC Instead, this reads Change Data Feed as a **plain batch DataFrame** (bounded start/end
# MAGIC version), which works fine under Spark Connect, and tracks "how far we've processed"
# MAGIC with a small control table instead of a streaming checkpoint. Re-running this notebook
# MAGIC is still idempotent: if no new Silver versions exist, the batch is empty and nothing
# MAGIC happens; if it's re-run after a partial failure, it recomputes the same version range
# MAGIC and MERGEs the same results back in.

# COMMAND ----------

PIPELINE_NAME = "gold_aggregates"
CONTROL_TABLE = f"{CATALOG}.{SCHEMA}.pipeline_state"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
    pipeline_name          STRING,
    last_processed_version  BIGINT,
    updated_ts               TIMESTAMP
) USING DELTA
""")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Helper: recompute aggregates only for the keys touched in this run
# MAGIC
# MAGIC Same logic as before, just called once against a batch (not streaming) DataFrame of
# MAGIC changed rows.

# COMMAND ----------

def upsert_gold(changes_df, run_id):
    key_rows = changes_df.select("transaction_date", "transaction_month").distinct().collect()

    if not key_rows:
        print(f"[gold run {run_id}] no changes")
        return

    changed_dates = sorted({r["transaction_date"] for r in key_rows})
    changed_months = sorted({r["transaction_month"] for r in key_rows})

    silver = spark.table(SILVER_TABLE)

    # --- Revenue by customer / day -----------------------------------------
    affected = silver.filter(F.col("transaction_date").isin(changed_dates))
    agg = (
        affected.groupBy("customer_id", "transaction_date")
        .agg(
            F.sum("amount").alias("gross_revenue"),
            F.sum("net_amount").alias("net_revenue"),
            F.count("*").alias("txn_count"),
            F.sum(F.col("is_refund").cast("int")).alias("refund_count"),
        )
        .withColumn("last_updated_ts", F.current_timestamp())
    )
    agg.createOrReplaceTempView("gold_customer_daily_updates")
    spark.sql(f"""
        MERGE INTO {GOLD_CUSTOMER_DAILY} t
        USING gold_customer_daily_updates s
        ON t.customer_id = s.customer_id AND t.transaction_date = s.transaction_date
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    # --- Revenue by month ----------------------------------------------------
    affected_m = silver.filter(F.col("transaction_month").isin(changed_months))
    agg_m = (
        affected_m.groupBy("transaction_month")
        .agg(
            F.sum("amount").alias("gross_revenue"),
            F.sum("net_amount").alias("net_revenue"),
            F.count("*").alias("txn_count"),
            F.countDistinct("customer_id").alias("distinct_customers"),
            F.sum(F.col("is_refund").cast("int")).alias("refund_count"),
        )
        .withColumn("last_updated_ts", F.current_timestamp())
    )
    agg_m.createOrReplaceTempView("gold_monthly_updates")
    spark.sql(f"""
        MERGE INTO {GOLD_MONTHLY} t
        USING gold_monthly_updates s
        ON t.transaction_month = s.transaction_month
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    # --- Category performance -------------------------------------------------
    agg_c = (
        affected_m.groupBy("category", "transaction_month")
        .agg(
            F.sum("net_amount").alias("net_revenue"),
            F.count("*").alias("txn_count"),
            F.round(F.avg("net_amount"), 2).alias("avg_txn_value"),
        )
        .withColumn("last_updated_ts", F.current_timestamp())
    )
    agg_c.createOrReplaceTempView("gold_category_updates")
    spark.sql(f"""
        MERGE INTO {GOLD_CATEGORY} t
        USING gold_category_updates s
        ON t.category = s.category AND t.transaction_month = s.transaction_month
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    print(f"[gold run {run_id}] refreshed {len(changed_dates)} day(s), {len(changed_months)} month(s)")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Run: read CDF as a bounded batch, process, then advance the control table
# MAGIC First run processes from version 0 through Silver's current latest version; subsequent
# MAGIC runs resume from `last_processed_version + 1`, so only genuinely new/changed rows are
# MAGIC reprocessed.

# COMMAND ----------

from delta.tables import DeltaTable

state_row = spark.sql(f"""
    SELECT last_processed_version FROM {CONTROL_TABLE} WHERE pipeline_name = '{PIPELINE_NAME}'
""").collect()
start_version = (state_row[0]["last_processed_version"] + 1) if state_row else 0

latest_version = (
    DeltaTable.forName(spark, SILVER_TABLE)
    .history(1)
    .select("version")
    .collect()[0]["version"]
)

if start_version > latest_version:
    print(f"Already up to date (last processed version {start_version - 1}, Silver at {latest_version}).")
else:
    changes_df = (
        spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", start_version)
        .option("endingVersion", latest_version)
        .table(SILVER_TABLE)
        .filter(F.col("_change_type").isin("insert", "update_postimage"))
    )

    upsert_gold(changes_df, run_id=f"{start_version}-{latest_version}")

    spark.sql(f"""
        MERGE INTO {CONTROL_TABLE} t
        USING (SELECT '{PIPELINE_NAME}' AS pipeline_name, {latest_version} AS last_processed_version,
                      current_timestamp() AS updated_ts) s
        ON t.pipeline_name = s.pipeline_name
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Processed Silver versions {start_version}-{latest_version}. Control table updated.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Reporting output

# COMMAND ----------

display(spark.sql(f"""
    SELECT * FROM {GOLD_MONTHLY} ORDER BY transaction_month
"""))

# COMMAND ----------

display(spark.sql(f"""
    SELECT * FROM {GOLD_CUSTOMER_DAILY}
    ORDER BY net_revenue DESC
    LIMIT 10
"""))

# COMMAND ----------

display(spark.sql(f"""
    SELECT * FROM {GOLD_CATEGORY}
    ORDER BY transaction_month, net_revenue DESC
"""))

In [0]:
spark.conf.set("spark.databricks.streaming.forEachBatch.optimized.enabled", "false")

In [0]:
dbutils.fs.rm("/Volumes/lakehouse_demo/transactions/checkpoints/gold_aggregates", recurse=True)

In [0]:
dbutils.fs.rm("/Volumes/lakehouse_demo/transactions/checkpoints/gold_aggregates", recurse=True)
